# SONAR VISION — treino customizado YOLO11n
Notebook preparado para GPU Colab, não executado neste ambiente. Não contrata serviços. Confira a cota da conta. Dados Bangkok são auxiliares; não validam códigos brasileiros.


## Fonte do projeto e armazenamento
Envie `artifacts/source/sonar-vision-source.zip` a `Meu Drive/SONAR/`. Autenticação do Drive é feita na interface do Colab; nunca cole credenciais no notebook.


In [ ]:
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

base = Path("/content/drive/MyDrive/SONAR")
root = base / "sonar-vision"
archive = base / "sonar-vision-source.zip"
if not (root / "pyproject.toml").exists():
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        if any(Path(n).is_absolute() or ".." in Path(n).parts for n in z.namelist()):
            raise ValueError("Arquivo ZIP contém caminhos inválidos")
        z.extractall(root)
os.chdir(root)
subprocess.run([sys.executable, "-m", "pip", "install", ".[vision]"], check=True)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA indisponível. Selecione um runtime GPU autorizado.")
print(torch.cuda.get_device_name(0))

## Amostra licenciada e separação
Amostra limitada a 120 famílias; não baixa vídeos. Os originais já podem conter aumentações do publicador. Sem IDs de sequência: resultados exploratórios. Não marque a divisão como revisada por vídeo sem evidências.


In [ ]:
from sonar_vision.training.dataset import verify_dataset

raw = root / "data/bangkok-pilot-raw"
data = root / "data/bangkok-pilot"
if not (raw / "manifest.csv").exists():
    cmd = [sys.executable, "scripts/download_bangkok_sample.py", "--count", "120"]
    if raw.exists():
        cmd.append("--resume")
    subprocess.run(cmd, check=True)
if not (data / "audit.json").exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "sonar_vision.cli.main",
            "prepare",
            "--manifest",
            str(raw / "manifest.csv"),
            "--output",
            str(data),
        ],
        check=True,
    )

audit = verify_dataset(data / "data.yaml")
print(audit["counts"], audit["split_assurance"])

## Smoke, treino e retomada
Runs são gravados no Drive. Não sobrescreva uma execução válida. O último checkpoint preserva estado de treino quando suportado pela biblioteca; um run finalizado não precisa ser retomado.


In [ ]:
from sonar_vision.training.runner import train

smoke = root / "artifacts/training/colab-smoke"
if not (smoke / "evidence.json").exists():
    train(data / "data.yaml", root / "configs/pilot.yaml", smoke, device="0", smoke=True)
run = root / "artifacts/training/colab-pilot-01"
evidence = run / "evidence.json"
previous = json.loads(evidence.read_text()) if evidence.exists() else {}
if previous.get("status") == "TRAINED":
    best = Path(previous["best"])
else:
    last = run / "run/weights/last.pt"
    if run.exists() and not last.exists():
        raise RuntimeError("Run incompleto sem checkpoint. Inspecione antes de continuar.")
    best = train(
        data / "data.yaml",
        root / "configs/pilot.yaml",
        run,
        device="0",
        resume=last if last.exists() else None,
    )
print("Checkpoint persistente:", best)

## Avaliação reservada
Execute uma vez após selecionar pelo conjunto de validação. Não ajuste hiperparâmetros usando este teste. Não são métricas SPTrans.


In [ ]:
from sonar_vision.training.runner import evaluate_detector

report = root / "reports/colab-evaluation"
metrics = report / "detector_metrics.json"
if metrics.exists():
    print(json.loads(metrics.read_text()))
else:
    print(evaluate_detector(best, data / "data.yaml", report, device="0"))

## Preservar artefatos
Guarde best.pt, last.pt, checkpoints periódicos, audit.json, manifesto, originais e evidências no Drive. Registre o caminho e o hash junto aos resultados do experimento. Restaure o modelo no computador de demonstração; não publique pesos/dados grandes no Git. Complete o teste brasileiro e a avaliação por encontro antes de declarar validação.
